# Kaggle Titantic Prediction Model: Primary Notebook
Conforms to the "Titantic Tutorial" notebook on Kaggle, as of April 14, 2026 (https://www.kaggle.com/code/alexisbcook/titanic-tutorial). This is run locally.

The competition is simple: use machine learning to create a model that predicts which passengers survived the Titanic shipwreck. In this challenge, we ask you to build a predictive model that answers the question: “what sorts of people were more likely to survive?” using passenger data (ie name, age, gender, socio-economic class, etc).


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Any results you write to the current directory are saved as output.

## Load the Data
Loads the data from the train.csv and test.csv files in the same directory and output a preview

In [2]:
train_data = pd.read_csv("train.csv")
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
test_data = pd.read_csv("test.csv")
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


## Explore Patterns
Test some very basic (and unlikely) patterns to get a feel for the data. Then, move into more complex patterns to train the model.

In [4]:
# Pattern 1: assumes that all female passengers survived (and all male passengers died.)

women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

% of women who survived: 0.7420382165605095


In [5]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

% of men who survived: 0.18890814558058924


## Construct a Model: Version 1
This will be a **random forest model**. Each tree will individually consider each passenger's data and vote on whether the individual survived. Then, the random forest model makes a democratic decision: the outcome with the most votes wins!

The code cell below constructs a model that makes a prediction based on the columns ("Pclass", "Sex", "SibSp", and "Parch"). It then saves these predictions in a CSV file called **submission.csv**, to be uploaded to Kaggle.

In [32]:
from sklearn.ensemble import RandomForestClassifier

y = train_data["Survived"]

features = ["Pclass", "Sex", "SibSp", "Parch"]
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

# 2. Force X_test to have the exact same columns as X
# This adds missing columns (as 0) and removes extra ones found only in test
X_test = X_test.reindex(columns=X.columns, fill_value=0)

model = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42, max_leaf_nodes = 500) # needs to test 5, 50, 500, 5000
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


### Getting Data on the Model's Performance

In [33]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import classification_report

predicted_survival = model.predict(X)
mean_absolute_error(y, predicted_survival) # MAE is not very useful for a classification model
print(classification_report(y, predicted_survival))


              precision    recall  f1-score   support

           0       0.82      0.90      0.86       549
           1       0.80      0.69      0.74       342

    accuracy                           0.82       891
   macro avg       0.81      0.79      0.80       891
weighted avg       0.82      0.82      0.81       891



In [58]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

# split data into training and validation data, for both features and target
# The split is based on a random number generator. Supplying a numeric value to
# the random_state argument guarantees we get the same split every time we
# run this script.
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 42)
# Define model
model = RandomForestClassifier(n_estimators=400, max_depth=3, random_state=42, max_leaf_nodes = 5000, max_features=3, min_samples_leaf=2, min_samples_split=5)
# Fit model
model.fit(train_X, train_y)

# get classification report on validation data
val_predictions = model.predict(val_X)
print("Unique in val_y:", val_y.unique())
print("Unique in predictions:", np.unique(val_predictions))
print(classification_report(val_y, val_predictions))

# Troubleshooting for accuracy stuck at 0
print("True labels:", val_y[:5])
print("Predicted labels:", val_X[:5])

probs = model.predict_proba(val_X)[:, 1]
# roc_curve(val_y, probs)
roc_auc_score(val_y, probs)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission_v2.csv', index=False)
print("Your submission was successfully saved!")


Unique in val_y: [1 0]
Unique in predictions: [0 1]
              precision    recall  f1-score   support

           0       0.82      0.89      0.85       134
           1       0.81      0.70      0.75        89

    accuracy                           0.81       223
   macro avg       0.81      0.79      0.80       223
weighted avg       0.81      0.81      0.81       223

True labels: 709    1
439    0
840    0
720    1
39     1
Name: Survived, dtype: int64
Predicted labels:      Pclass  SibSp  Parch  Sex_female  Sex_male
709       3      1      1       False      True
439       2      0      0       False      True
840       3      0      0       False      True
720       2      0      1        True     False
39        3      1      0        True     False
Your submission was successfully saved!


### Fine-Tuning the Model

In [56]:
# Compare mean absolute error scores when we change the value for max_leaf_nodes
# max_leaf_nodes allows us to control underfitting vs overfitting

# from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
# using GridSearchCV to find the best settings for the model using k-fold cross-validation
param_grid = {'n_estimators': [300, 400, 600], 'max_depth': [None, 2, 3, 5, 10], 'max_features': [3, 6, 10, 12], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 5, 10]}
grid_search = GridSearchCV(RandomForestClassifier(), param_grid, cv=3, n_jobs = -1)
grid_search.fit(train_X, train_y)
print(f"Best Params: {grid_search.best_params_}")

# def get_accuracy(max_leaf_nodes, train_X, val_X, train_y, val_y):
#     model = RandomForestClassifier(max_leaf_nodes=max_leaf_nodes, random_state=0)
#     model.fit(train_X, train_y)
#     preds_val = model.predict(val_X)
#     accuracy = accuracy_score(val_y, preds_val)
#     return(accuracy)

# # compare MAE with differing values of max_leaf_nodes
# for max_leaf_nodes in [5, 50, 500, 5000]:
#     my_accuracy = get_accuracy(max_leaf_nodes, train_X, val_X, train_y, val_y) # expected 2D array, got 1D array instead???
#     print("Max leaf nodes: %d  \t\t Accuracy:  %d" %(max_leaf_nodes, my_accuracy))

Best Params: {'max_depth': 3, 'max_features': 3, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 400}


## Version 2
This is an improved version of the model above, with some parameters fine-tuned and updated.